# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('../data/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [13]:
# Task 1
# YOUR CODE HERE

df_fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()

region_fossil  = df_fossil.groupby('Region')['TWh'].sum().sort_values(ascending=False)
country_fossil = df_fossil.groupby('Country')['TWh'].sum().sort_values(ascending=False)
top_region, top_region_twh   = region_fossil.index[0], region_fossil.iloc[0]
top_country, top_country_twh = country_fossil.index[0], country_fossil.iloc[0]
top_region_share = top_region_twh / df_fossil['TWh'].sum()

fossil_colors = {
    'Coal':        '#E07B39',   
    'Oil':         '#2E75B6',   
    'Natural Gas': '#F5C842',   
}

fig = px.treemap(
    df_fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map=fossil_colors,
    labels={'Source': 'Fossil source', 'TWh': 'Energy (TWh)'},
    height=700, width=1100,
    template='plotly_white',
)

fig.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',   # TWh only — no percentages
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>',
)

fig.update_layout(
    title=dict(
        text=(f'{top_region} burns {top_region_share:.0%} of all fossil energy ({top_region_twh:,.0f} TWh) — '
              f'but {top_country} is the largest single consumer at {top_country_twh:,.0f} TWh'),
        x=0, xanchor='left',
    ),
    font=dict(family='Arial', size=12),
)

fig.data[0].marker.colors = [
    c if c in fossil_colors.values() else '#DDDDDD'
    for c in fig.data[0].marker.colors
]

fig.show()

## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [3]:
# Task 2 — charts
# YOUR CODE HERE

df_lowc = (df.loc[df['Source_Type'] == 'Low-carbon']
             .groupby('Country', as_index=False)['TWh'].sum()
             .sort_values('TWh', ascending=False))
df_lowc['All'] = 'Low-carbon'         

leader, leader_twh = df_lowc.iloc[0]['Country'], df_lowc.iloc[0]['TWh']
runner_up = df_lowc.iloc[1]['Country']

fig_tm = px.treemap(
    df_lowc,
    path=['All', 'Country'],
    values='TWh',
    color_discrete_sequence=['#2E75B6'],     
    height=550, width=1000,
    template='plotly_white',
)
fig_tm.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>',
    root_color='#DDDDDD',                  
)
fig_tm.update_layout(
    title=dict(text='Low-carbon energy (nuclear + hydro) by country, TWh', x=0, xanchor='left'),
    font=dict(family='Arial', size=12),
)
fig_tm.show()

df_lowc['Highlight'] = np.where(df_lowc['Country'] == leader, 'Leader', 'Other')

fig_bar = px.bar(
    df_lowc,
    x='TWh', y='Country',
    orientation='h',
    color='Highlight',
    color_discrete_map={'Leader': '#2E75B6', 'Other': '#AAAAAA'},
    text='TWh',
    category_orders={'Country': df_lowc['Country'].tolist()},   
    labels={'TWh': 'Low-carbon energy (TWh)'},
    height=550, width=1000,
    template='plotly_white',
)
fig_bar.update_traces(
    texttemplate='%{x:,.0f} TWh', textposition='outside',
    hovertemplate='<b>%{y}</b><br>%{x:,.0f} TWh<extra></extra>',
    cliponaxis=False,
)
fig_bar.update_layout(
    title=dict(text=f'{leader} leads low-carbon energy with {leader_twh:,.0f} TWh, ahead of {runner_up}',
               x=0, xanchor='left'),
    font=dict(family='Arial', size=12),
    showlegend=False,
    xaxis=dict(visible=False),
    yaxis=dict(title=''),
    bargap=0.25,
)
fig_bar.show()

### Task 2 — reflection

**Which chart tells the ranking story more clearly, and why?**

The horizontal bar chart tells the ranking story more clearly. Every bar starts from the same place, so our eye simply compares the lengths of each bar. This way we can judge far more accurately than the areas a treemap uses. Also sorting the output make the ranking clear: France at the top, then Norway, Canada and Brazil, and anyone can read the positions like "Germany is above Mexico" without counting. In the treemap the rectangles have different shapes (some wide, some tall), so countries of similar size, such as Germany (929 TWh) and Mexico (906), are hard to tell which one is bigger.

## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

In [14]:
# Task 3
# YOUR CODE HERE

region_mix = (df_fossil.groupby(['Region', 'Source'])['TWh'].sum()
                       .unstack(fill_value=0)        
                       .stack().rename('TWh').reset_index())
region_mix['Share'] = region_mix['TWh'] / region_mix.groupby('Region')['TWh'].transform('sum')

coal = region_mix.loc[region_mix['Source'] == 'Coal'].sort_values('Share', ascending=False)
region_order = coal['Region'].tolist()
top_coal_region, top_coal_share = coal.iloc[0]['Region'], coal.iloc[0]['Share']

region_mix['Label'] = np.where((region_mix['Source'] == 'Coal') & (region_mix['Share'] > 0),
                               region_mix['Share'].map('{:.0%}'.format), '')

coal_accent = {
    'Coal':        fossil_colors['Coal'],   
    'Oil':         '#BBBBBB',              
    'Natural Gas': '#DDDDDD',             
}

fig = px.bar(
    region_mix,
    x='Share', y='Region',
    orientation='h',
    color='Source',
    text='Label',
    color_discrete_map=coal_accent,
    category_orders={'Region': region_order,
                     'Source': ['Coal', 'Oil', 'Natural Gas']},   
    labels={'Share': 'Share of fossil energy', 'Source': 'Fossil source'},
    hover_data={'Label': False, 'Share': ':.0%', 'TWh': ':,.0f'},
    height=550, width=1000,
    template='plotly_white',
)

fig.update_traces(
    textposition='inside', insidetextanchor='start',
    textfont=dict(color='white', size=12),
    marker_line_color='white', marker_line_width=1,
)

fig.update_layout(
    title=dict(text=f'{top_coal_region} relies most on coal: {top_coal_share:.0%} of its fossil energy comes from coal',
               x=0, xanchor='left'),
    font=dict(family='Arial', size=12),
    bargap=0.25,
    legend=dict(orientation='h', x=0, y=1.08, title=''),
    xaxis=dict(visible=False, range=[0, 1]),
    yaxis=dict(title=''),
)

fig.add_annotation(x=1, y=0, xref='x', yref='paper', xanchor='right', yanchor='top',
                   yshift=-4, text='100%', showarrow=False, font=dict(size=11, color='grey'))

fig.show()

### Task 3 — reflection

**Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.**

- **Treemap** "How much fossil energy does each region and country use in total, and which countries are include in each region?" The treemap shows total use of each Region and Country, so you can see that Asia is the largest fossil consumer and that Japan, China, South Korea and India are included in that region. The stacked bar scales every region to the same length, so all information about total volume and individual countries is gone.

- **Stacked bar** "Which region depends most on coal relative to its other fossil fuels, and how do regions rank on that?" Every bar shares the same baseline and is sorted by coal share, so anyone can read that Africa gets 78% of its fossil energy from coal, compared with 49% for Asia, and rank all seven regions. In the treemap, coal's share is hidden inside rectangles of different sizes and shapes, so comparing with eye across regions isn't possible.